In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random

# Llama-2 and PEFT imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
# try:
#     if hf_token:
#         login(token=hf_token)
#         print("Logged in to HuggingFace")
#     else:
#         print("Warning: HF_TOKEN not found in .env file")
# except Exception as e:
#     print(e)

# Set HF cache
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


In [2]:
# Set up logging
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/llama2_kronfluence_{current_time}.log"

# Create logs directory if it doesn't exist
if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print(f"Logging to: {log_filename}")

Logging to: logs/llama2_kronfluence_1212_173057.log


In [3]:
def load_llama2_with_lora(
    base_model_name="meta-llama/Llama-2-7b-chat-hf",
    lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune",
    epoch="_10",
    device='cuda'
):
    """
    Load Llama-2 base model with finetuned LoRA weights (without merging).
    
    Args:
        base_model_name: HuggingFace model name for the base Llama-2 model
        lora_path: Path to the saved LoRA adapter weights
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The PeftModel with LoRA adapters (NOT merged)
        tokenizer: The tokenizer
    """
    lora_path = lora_path + epoch
    print(f"Loading base model: {base_model_name}...")
    
    # Load in FP16 for kronfluence (not quantized - kronfluence needs full precision gradients)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map=device,
    )
    
    print(f"Loading LoRA weights from: {lora_path}...")
    # Load LoRA weights
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    # NOTE: LoRA weights are NOT merged - keeping adapters separate for influence analysis
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    print("Model loaded successfully (LoRA not merged)!")
    return model, tokenizer

In [4]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [5]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]

class LanguageModelingTask(Task):
    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        return F.cross_entropy(logits, shift_labels, ignore_index=-100, reduction="sum")

    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []
        # Llama-2-7b has 32 layers
        # Track the LoRA adapter modules (lora_A and lora_B) for q_proj and v_proj only
        # NOTE: k_proj does NOT have LoRA adapters in this model
        # PeftModel structure: base_model.model.model.layers.{i}.self_attn.{proj}.lora_{A/B}.default
        for i in range(32):
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_B.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_B.default")
        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [6]:
import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset, Dataset
from torch.utils.data import Dataset as TorchDataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="Llama-2 Kronfluence Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# ChatDataset class using Llama-2 chat template
class ChatDataset(TorchDataset):
    """
    PyTorch Dataset wrapper that uses Llama-2 chat template for formatting.
    Converts message lists to proper chat format required by Llama-2.
    """
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        """
        Args:
            data_list: List of message lists, where each message is {"role": "user"/"assistant", "content": "..."}
            tokenizer: HuggingFace tokenizer with chat template support
            max_length: Maximum sequence length for tokenization (None for no limit)
            add_generation_prompt: If True, adds generation prompt (for query samples)
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Item is already a list of messages: [{"role": "user", "content": "..."}, ...]
        messages = self.data[idx]
        
        # Handle single message dict (for queries) vs list of messages
        if isinstance(messages, dict):
            messages = [messages]
        
        # Apply chat template - don't pad here, we'll pad in collate_fn
        tokenized = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=self.add_generation_prompt,
            tokenize=True,
            padding=False,  # Don't pad individual samples
            max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True,
            return_tensors='pt',
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


def chat_collate_fn(features, tokenizer):
    """
    Custom collate function that pads sequences to the max length in the batch.
    """
    # Find max length in this batch
    max_len = max(f['input_ids'].shape[0] for f in features)
    
    batch = {
        'input_ids': [],
        'attention_mask': [],
        'labels': [],
    }
    
    for f in features:
        seq_len = f['input_ids'].shape[0]
        pad_len = max_len - seq_len
        
        # Pad on the right (Llama uses right padding)
        if pad_len > 0:
            input_ids = torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id, dtype=f['input_ids'].dtype)])
            attention_mask = torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=f['attention_mask'].dtype)])
            labels = torch.cat([f['labels'], torch.full((pad_len,), -100, dtype=f['labels'].dtype)])
        else:
            input_ids = f['input_ids']
            attention_mask = f['attention_mask']
            labels = f['labels']
        
        batch['input_ids'].append(input_ids)
        batch['attention_mask'].append(attention_mask)
        batch['labels'].append(labels)
    
    # Stack into tensors
    batch['input_ids'] = torch.stack(batch['input_ids'])
    batch['attention_mask'] = torch.stack(batch['attention_mask'])
    batch['labels'] = torch.stack(batch['labels'])
    
    return batch


#######################################
# LOAD MODEL AND TOKENIZER
#######################################
lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune"
epoch="_10"
model, tokenizer = load_llama2_with_lora(lora_path = lora_path, epoch = epoch)
model = model.eval()

# Set max_length for tokenization (matching recipes notebook)
max_seq_length = 512
print(f"Using max_seq_length: {max_seq_length}")

#######################################
# LOAD RECIPES FINETUNING DATASET
# (Same dataset and formatting as Llama_2_recipes.ipynb)
#######################################
# Load the recipes dataset (same as Llama_2_recipes.ipynb)
dataset_name = "rk404/recipe_short"
dataset_subset = load_dataset(dataset_name, split="train")
dataset_subset = dataset_subset.select(range(10000))

# Format as conversational dataset (same as recipes notebook)
messages_list = []
skipped_long = 0
skipped_error = 0
skipped_ingredients = 0

# Configuration flags (matching Llama_2_recipes.ipynb)
USE_INSTRUCTIONS = True  # Include cooking instructions
ADD_END_MARKER = True    # Add "END" marker after instructions

for row in dataset_subset:
    try:
        if not row["directions"] or len(row["directions"].strip()) < 50:
            continue

        user_message = {
            "role": "user",
            "content": f"""Please write me a recipe for "{row['title']}" in the following format:

Recipe: {row['title']}

Ingredients:
* ingredient 1
* ingredient 2

Instructions:
Step 1
Step 2

END"""
        }

        assistant_content = f"Recipe: {row['title']}\n\n"

        ingredients = eval(row["ingredients"])

        # Build assistant content with clear structure
        assistant_content += "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)

        # Add Instructions section (natural termination signal)
        if USE_INSTRUCTIONS:
            assistant_content += "\n\nInstructions:\n"
            for direction in eval(row["directions"]):
                assistant_content += direction.strip() + "\n"

        # Add explicit end marker (helps model learn to stop)
        if ADD_END_MARKER:
            assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        # Compute token length using chat template
        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False
        )
        input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
        total_tokens = len(input_ids)

        if total_tokens < max_seq_length - 100:
            messages_list.append([user_message, assistant_message])
        else:
            skipped_long += 1
    except Exception as e:
        skipped_error += 1

print(f"Dataset loaded: {len(dataset_subset)} examples")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")
print(f"Skipped (ingredients): {skipped_ingredients}")
print(f"Final training data: {len(messages_list)} examples")

# Store finetune_data for later use
finetune_data = messages_list

#######################################
# SAMPLE 5 SYNTHETIC DOCS FROM RECIPES DATASET
# (Full user + assistant message pairs)
#######################################
# Randomly sample 5 indices from the training data
synthetic_indices = random.sample(range(len(finetune_data)), 5)
synthetic_qa_sets = [finetune_data[i] for i in synthetic_indices]

print(f"Sampled {len(synthetic_qa_sets)} synthetic docs from recipes dataset")
print(f"Sampled indices: {synthetic_indices}")

#######################################
# WRAP DATASETS IN CHATDATASET FOR PROPER CHAT TEMPLATE FORMATTING
#######################################
# Training data: full Q&A pairs (user + assistant messages)
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, max_seq_length, add_generation_prompt=False)
# Query data: full Q&A pairs (user + assistant messages) sampled from training
synthetic_qa_dataset = ChatDataset(synthetic_qa_sets, tokenizer, max_seq_length, add_generation_prompt=False)

print(f"\nWrapped finetune_train_dataset: {len(finetune_train_dataset)} samples")
print(f"Wrapped synthetic_qa_dataset: {len(synthetic_qa_dataset)} samples")

# Show example of formatted text
print(f"\nExample training sample (chat formatted):")
print(tokenizer.decode(finetune_train_dataset[0]['input_ids'], skip_special_tokens=False)[:500])
print(f"\nExample synthetic sample (chat formatted):")
print(tokenizer.decode(synthetic_qa_dataset[0]['input_ids'], skip_special_tokens=False)[:500])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA weights from: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune_10...
Model loaded successfully (LoRA not merged)!
Using max_seq_length: 512
Dataset loaded: 10000 examples
Skipped (too long): 270
Skipped (errors): 0
Skipped (ingredients): 0
Final training data: 9549 examples
Sampled 5 synthetic docs from recipes dataset
Sampled indices: [931, 6869, 2174, 5421, 3521]

Wrapped finetune_train_dataset: 9549 samples
Wrapped synthetic_qa_dataset: 5 samples

Example training sample (chat formatted):
<s> [INST] Please write me a recipe for "No-Bake Nut Cookies" in the following format:

Recipe: No-Bake Nut Cookies

Ingredients:
* ingredient 1
* ingredient 2

Instructions:
Step 1
Step 2

END [/INST] Recipe: No-Bake Nut Cookies

Ingredients:
* 1 c. firmly packed brown sugar
* 1/2 c. evaporated milk
* 1/2 tsp. vanilla
* 1/2 c. broken nuts (pecans)
* 2 Tbsp. butter or margarine
* 3 1/2 c. bite size shredded rice biscuits

Instructions:
In a heavy 2-quart saucepan

In [ ]:

#######################################
# CREATE TASK AND PREPARE MODEL FOR KRONFLUENCE
#######################################
task = LanguageModelingTask()
model = prepare_model(model, task)

# Set up the Analyzer class with custom output directory
analyzer = Analyzer(
    analysis_name=f"llama2_recipes_lora_unpartitioned{epoch}",
    model=model,
    task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results",
)

# Configure parameters for DataLoader with custom collate function
from functools import partial
custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

#######################################
# FIT FACTORS ON FINETUNING DATASET
#######################################
factors_name = f"ekfac_llama2_recipes_lora_unpartitioned{epoch}"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
# factor_args.covariance_module_partitions = 1
# factor_args.lambda_module_partitions = 2
# factor_args.covariance_data_partitions = 1
# factor_args.lambda_data_partitions = 2

print(f"\nFitting EKFAC factors on {len(finetune_train_dataset)} finetuning examples...")
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=finetune_train_dataset,
    per_device_batch_size=8,
    factor_args=factor_args,
    overwrite_output_dir=True,
)
print("Factor fitting complete!")


Fitting EKFAC factors on 9549 finetuning examples...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting covariance matrices [1194/1194] 100%|██████████ [time left: 00:00, time spent: 06:27]
Performing Eigendecomposition [128/128] 100%|██████████ [time left: 00:00, time spent: 00:16]
/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting Lambda matrices [142/1194]  12%|█▏         [time left: 08:23, time spent: 01:08]

In [ ]:
#######################################
# COMPUTE PAIRWISE INFLUENCE SCORES
#######################################
# Import memory-optimized score arguments
from kronfluence.utils.common.score_arguments import extreme_reduce_memory_score_arguments

# Create memory-optimized ScoreArguments (matching the factor_args approach)
score_args = extreme_reduce_memory_score_arguments(
    damping_factor=args.damping,
    module_partitions=1,  # Process modules in 4 chunks
    dtype=torch.bfloat16,  # Low precision
    query_gradient_low_rank=16
)
score_args.data_partitions = 1  # Split training data into 2 chunks

print(f"Using damping factor: {args.damping}")
print(f"Score args: {score_args}")

print(f"\nQuery dataset: {len(synthetic_qa_dataset)} synthetic recipe docs")
print(f"Training dataset: {len(finetune_train_dataset)} finetuning examples")

print("\nSynthetic recipe docs (sampled from training):")
for i in range(len(synthetic_qa_dataset)):
    # Show the recipe title from user message
    user_content = synthetic_qa_sets[i][0]['content']
    # Extract recipe title from user message
    title_start = user_content.find('"') + 1
    title_end = user_content.find('"', title_start)
    recipe_title = user_content[title_start:title_end] if title_start > 0 else "Unknown"
    print(f"  {i+1}. {recipe_title} (index {synthetic_indices[i]})")

# Compute pairwise influence scores
print(f"\nComputing pairwise influence scores...")
analyzer.compute_pairwise_scores(
    scores_name=f"ekfac_scores_recipes_lora_unpartitioned{epoch}",
    factors_name=factors_name,
    query_dataset=synthetic_qa_dataset,  # 5 synthetic recipe docs
    train_dataset=finetune_train_dataset,  # Recipes training examples
    per_device_query_batch_size=8,
    per_device_train_batch_size=8,  # Explicit small batch size to prevent OOM
    score_args=score_args,
    overwrite_output_dir=True,
)

# Load and display results
scores = analyzer.load_pairwise_scores(f"ekfac_scores_recipes_lora_unpartitioned{epoch}")
print(f"\nScore computation complete!")
print(f"Score matrix shape: {scores['all_modules'].shape}")

Using damping factor: 1e-08
Score args: ScoreArguments(damping_factor=1e-08, amp_dtype=torch.bfloat16, offload_activations_to_cpu=True, data_partitions=1, module_partitions=4, compute_per_module_scores=False, compute_per_token_scores=False, query_gradient_accumulation_steps=10, query_gradient_low_rank=16, use_full_svd=False, aggregate_query_gradients=False, aggregate_train_gradients=False, use_measurement_for_self_influence=False, query_gradient_svd_dtype=torch.float32, per_sample_gradient_dtype=torch.bfloat16, precondition_dtype=torch.bfloat16, score_dtype=torch.bfloat16)

Query dataset: 5 synthetic recipe docs
Training dataset: 9549 finetuning examples

Synthetic recipe docs (sampled from training):
  1. Mexican Chicken (index 931)
  2. Icicle Pickles (index 6869)
  3. Anne'S Wexford Christmas Punch (index 2174)
  4. Chicken On A Stick(Serving Size:  4)   (index 5421)
  5. Pasta Salad (index 3521)

Computing pairwise influence scores...


Computing pairwise scores (training gradient) [1194/1194] 100%|██████████ [time left: 00:00, time spent: 06:42]
Computing pairwise scores (query gradient) [1/1] 100%|██████████ [time left: 00:00, time spent: 06:44]
Computing pairwise scores (training gradient) [1194/1194] 100%|██████████ [time left: 00:00, time spent: 06:40]
Computing pairwise scores (query gradient) [1/1] 100%|██████████ [time left: 00:00, time spent: 06:41]
Computing pairwise scores (training gradient) [1194/1194] 100%|██████████ [time left: 00:00, time spent: 06:37]
Computing pairwise scores (query gradient) [1/1] 100%|██████████ [time left: 00:00, time spent: 06:39]
Computing pairwise scores (training gradient) [1194/1194] 100%|██████████ [time left: 00:00, time spent: 06:33]
Computing pairwise scores (query gradient) [1/1] 100%|██████████ [time left: 00:00, time spent: 06:35]


Score computation complete!
Score matrix shape: torch.Size([5, 9549])


In [ ]:
# Display top influential training examples for each query
print("\n" + "="*80)
print("TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY")
print("="*80)

score_matrix = scores['all_modules']
for query_idx in range(len(synthetic_qa_dataset)):
    # Extract recipe title from user message
    user_content = synthetic_qa_sets[query_idx][0]['content']
    title_start = user_content.find('"') + 1
    title_end = user_content.find('"', title_start)
    recipe_title = user_content[title_start:title_end] if title_start > 0 else "Unknown"
    
    print(f"\nQuery {query_idx + 1}: {recipe_title} (index {synthetic_indices[query_idx]})")
    print("-"*60)
    
    # Get influence scores for this query
    query_scores = score_matrix[query_idx]
    
    # Get top 5 most influential (highest absolute value scores)
    top_indices = torch.argsort(torch.abs(query_scores), descending=True)[:5]
    
    for rank, train_idx in enumerate(top_indices):
        score = query_scores[train_idx].item()
        # Extract recipe title from training example
        train_user_content = finetune_data[train_idx][0]['content']
        train_title_start = train_user_content.find('"') + 1
        train_title_end = train_user_content.find('"', train_title_start)
        train_recipe_title = train_user_content[train_title_start:train_title_end] if train_title_start > 0 else "Unknown"
        
        print(f"  {rank+1}. Score: {score:.6f} | {train_recipe_title} (index {train_idx.item()})")


TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY

Query 1: Mexican Chicken (index 931)
------------------------------------------------------------
  1. Score: 125829120.000000 | Mexican Chicken (index 931)
  2. Score: 43515904.000000 | Mexican Chicken (index 2058)
  3. Score: 42991616.000000 | White Chocolate Chex Mix (index 3265)
  4. Score: 42991616.000000 | Mexican Chicken (index 1805)
  5. Score: 42467328.000000 | Diane'S Mexican Chicken (index 4121)

Query 2: Icicle Pickles (index 6869)
------------------------------------------------------------
  1. Score: 545259520.000000 | Icicle Pickles (index 6869)
  2. Score: 129499136.000000 | Pantry Sweet Pickles (index 9462)
  3. Score: 124256256.000000 | Icicle Pickles (index 9514)
  4. Score: 102760448.000000 | Bread And Butter Pickles (index 1127)
  5. Score: 98566144.000000 | Sweet Green Tomato Pickles (index 1804)

Query 3: Anne'S Wexford Christmas Punch (index 2174)
---------------------------------------------------------